In [1]:
# Check GPU
!nvidia-smi

Fri Jan 16 07:01:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             33W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip uninstall -y transformers peft accelerate

Found existing installation: transformers 4.41.0
Uninstalling transformers-4.41.0:
  Successfully uninstalled transformers-4.41.0
Found existing installation: peft 0.7.0
Uninstalling peft-0.7.0:
  Successfully uninstalled peft-0.7.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0


In [3]:
# Install dependencies
!pip install -q transformers peft accelerate evaluate datasets bitsandbytes

In [5]:
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_from_disk
import torch
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

2026-01-16 07:02:09.512080: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768546929.536351     297 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768546929.543631     297 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768546929.562596     297 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768546929.562626     297 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768546929.562629     297 computation_placer.cc:177] computation placer alr

PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB


In [12]:
# CONFIG 
MODEL_NAME = "xlm-roberta-base"
MAX_SAMPLES = 40000  
EPOCHS = 5
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
MAX_LENGTH = 384
DOC_STRIDE = 128
LORA_R = 16
LORA_ALPHA = 32

DATASET_PATH = "/kaggle/input/squad-normalized-for-xlm-roberta/squad_normalized/squad_normalized"  
OUTPUT_DIR = "/kaggle/working/stage1_output"
CHECKPOINT_DIR = "/kaggle/working/stage1_checkpoints"
FINAL_MODEL_DIR = "/kaggle/working/stage1_best"

## Load Dataset

In [7]:
print(f"Loading SQuAD from {DATASET_PATH}")
squad = load_from_disk(DATASET_PATH)
print(f"Original train size: {len(squad['train'])}")
print(f"Original validation size: {len(squad['validation'])}")

# Subsample for faster training
if MAX_SAMPLES < len(squad['train']):
    squad['train'] = squad['train'].shuffle(seed=42, keep_in_memory=True).select(range(MAX_SAMPLES), keep_in_memory=True)
    print(f"Subsampled train size: {len(squad['train'])}")

# Subsample validation too
squad['validation'] = squad['validation'].select(range(min(2000, len(squad['validation']))), keep_in_memory=True)
print(f"Validation size: {len(squad['validation'])}")

Loading SQuAD from /kaggle/input/squad-normalized-for-xlm-roberta/squad_normalized/squad_normalized
Original train size: 130319
Original validation size: 11873
Subsampled train size: 40000
Validation size: 2000


## Load Model & Tokenizer

In [8]:
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

# Apply LoRA
print(f"Applying LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
lora_config = LoraConfig(
    task_type=TaskType.QUESTION_ANS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.1,
    target_modules=["query", "value", "key"],
    bias="none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading tokenizer: xlm-roberta-base
Loading model: xlm-roberta-base


Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Applying LoRA: r=16, alpha=32
trainable params: 886,274 || all params: 278,340,868 || trainable%: 0.3184131767527577


## Prepare Data

In [9]:
def prepare_train_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    
    tokenized["start_positions"] = []
    tokenized["end_positions"] = []
    
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        
        # Nếu không có answer, đặt vị trí là CLS token
        if len(answers["answer_start"]) == 0:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
            continue
        
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        
        # Lấy sequence_ids để xác định context
        sequence_ids = tokenized.sequence_ids(i)
        
        # Tìm vị trí bắt đầu và kết thúc của context
        context_start = sequence_ids.index(1) if 1 in sequence_ids else 0
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1) if 1 in sequence_ids else len(sequence_ids)
        
        # Tìm token start index
        token_start_index = context_start
        while token_start_index <= context_end and offsets[token_start_index][0] <= start_char:
            token_start_index += 1
        token_start_index -= 1
        
        # Tìm token end index
        token_end_index = context_end
        while token_end_index >= context_start and offsets[token_end_index][1] >= end_char:
            token_end_index -= 1
        token_end_index += 1
        
        if (token_start_index < context_start or 
            token_end_index > context_end or
            token_start_index >= len(offsets) or
            token_end_index >= len(offsets) or
            token_start_index < 0 or
            token_end_index < 0):
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        elif not (offsets[token_start_index][0] <= start_char and 
                  offsets[token_end_index][1] >= end_char):
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        else:
            tokenized["start_positions"].append(token_start_index)
            tokenized["end_positions"].append(token_end_index)
    
    return tokenized

print("Tokenizing datasets")
train_dataset = squad['train'].map(
    prepare_train_features,
    batched=True,
    remove_columns=squad['train'].column_names,
    desc="Tokenizing train",
    keep_in_memory=True  # Thêm này để tránh lỗi read-only
)

val_dataset = squad['validation'].map(
    prepare_train_features,
    batched=True,
    remove_columns=squad['validation'].column_names,
    desc="Tokenizing validation",
    keep_in_memory=True  # Thêm này để tránh lỗi read-only
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

Tokenizing datasets


Tokenizing train:   0%|          | 0/40000 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/2000 [00:00<?, ? examples/s]

Train dataset: 40918 samples
Val dataset: 2008 samples


## Training

In [13]:
# Sau khi restart runtime, chạy lại từ đầu

from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_from_disk
import torch

# load dataset và tokenize 
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="steps",  
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=3,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_accumulation_steps=4,
    dataloader_num_workers=2,
    report_to="none",
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting Stage 1 Training (EN Warm-up)")
trainer.train()

Starting Stage 1 Training (EN Warm-up)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss
1000,2.616000,1.936828
2000,2.243600,1.882795
3000,2.035800,1.754378
4000,1.933500,1.710882
5000,1.872800,1.613884
6000,1.876700,1.669916


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

TrainOutput(global_step=6390, training_loss=2.1983626053739975, metrics={'train_runtime': 6829.1722, 'train_samples_per_second': 29.958, 'train_steps_per_second': 0.936, 'total_flos': 4.048842960521626e+16, 'train_loss': 2.1983626053739975, 'epoch': 4.997067448680352})

## Save Best Model

In [14]:
print(f"Saving best model to {FINAL_MODEL_DIR}")
print("Note: Trainer has already loaded the best checkpoint (lowest eval_loss)")
print(f"Best model will be saved to: {FINAL_MODEL_DIR}")

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Best model saved to: {FINAL_MODEL_DIR}")


Saving best model to /kaggle/working/stage1_best
Note: Trainer has already loaded the best checkpoint (lowest eval_loss)
Best model will be saved to: /kaggle/working/stage1_best
Best model saved to: /kaggle/working/stage1_best


In [15]:
# Check output files
!ls -lh /kaggle/working/stage1_best/

total 25M
-rw-r--r-- 1 root root  580 Jan 16 09:54 adapter_config.json
-rw-r--r-- 1 root root 3.4M Jan 16 09:54 adapter_model.safetensors
-rw-r--r-- 1 root root 5.0K Jan 16 09:54 README.md
-rw-r--r-- 1 root root 4.9M Jan 16 09:54 sentencepiece.bpe.model
-rw-r--r-- 1 root root  280 Jan 16 09:54 special_tokens_map.json
-rw-r--r-- 1 root root 1.2K Jan 16 09:54 tokenizer_config.json
-rw-r--r-- 1 root root  17M Jan 16 09:54 tokenizer.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
